In [35]:
from lib import read_parquet_user, read_parquet_item, read_parquet_purchase, split_and_save_parquet, build_feature_label

#### Read dataframes

In [36]:
item_df = read_parquet_item('.././preprocessed-feature/')
item_df.head()

item_id,price,category_l1,category,brand_final,target_user_group_final,item_type_final,sale_status,age_bucket_final,price_norm
str,"decimal[38,4]",str,str,str,str,str,i32,str,f64
"""0502020000004""",99000.0000,"""babycare""","""núm ty dr brown""","""dr.brown's""","""sơ sinh""",null,0,"""0-6m""",-0.178933
"""0010290040150""",69000.0000,"""thời trang""","""bộ quần áo bé gái""","""con cưng""","""bé gái""","""bộ quần áo""",0,"""2-4y""",-0.237627
"""0008010000015""",45000.0000,"""đồ chơi & sách""","""gặm nướu khác""","""thương hiệu khác""","""bé trai""",null,0,"""1-4y""",-0.284582
"""0020010000094""",401000.0000,"""tã""","""merries_sơ sinh""","""merries nhật""","""sơ sinh""",null,0,"""0-6m""",0.411922
"""0020010000098""",401000.0000,"""tã""","""merries_tã quần""","""merries nhật""","""sơ sinh""",null,0,"""6-12m""",0.411922


In [37]:
user_df = read_parquet_user('.././preprocessed-feature/')
user_df.head()

customer_id,gender,location,province,membership,region,location_name,install_app,district,milk_segment_preference,diaper_segment_preference
i32,str,i32,str,str,str,str,str,str,i64,i64
2102259,"""Nữ""",838,"""Kiên Giang""","""Standard""","""Đồng bằng sông Cửu Long""","""KGI - Lô L10-12 QL61""","""In-Store""","""Gò Quao""",null,null
2102190,"""Nữ""",728,"""Gia Lai""","""Standard""","""Tây Nguyên""","""GLA - 113 Hai Bà Trưng""","""In-Store""","""Pleiku""",null,2
2102266,"""Nữ""",535,"""Đồng Nai""","""Standard""","""Đông Nam Bộ""","""DON - 537 Cách Mạng Tháng Tám""","""In-Store""","""Biên Hòa""",null,null
2102268,"""Nam""",545,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 304B Trường Chinh""","""In-Store""","""Tân Bình""",null,null
2102273,"""Nữ""",365,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 266A Tỉnh Lộ 15""","""In-Store""","""Củ Chi""",null,null


In [38]:
transaction_df = read_parquet_purchase('.././preprocessed-feature/')
transaction_df.head()

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level,avg_transaction_amount_per_purchase,segment_name,avg_cat_l1_per_purchase,segment_name_right
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str,f64,str,f64,str
"""2803000000013""",2,6604637,2024-10-14 18:02:15.860,578,44100.0,10.694238,0.1,"""In-Store""","""card""",360d 22h 52m 10s 780ms,10,"""Autumn""","""High""",580106.153069,"""Trung cấp""",1.686047,"""Mua vừa"""
"""0020020000172""",1,6594820,2024-10-14 07:37:39.637,831,30000.0,10.308986,0.0,"""In-Store""","""cash""",352d 10h 35m 46s 53ms,10,"""Autumn""","""High""",274824.47673,"""Bình dân""",1.895522,"""Mua vừa"""
"""0020120000014""",1,7957733,2024-10-14 14:50:28.070,63,80500.0,11.296025,0.3,"""In-Store""","""cash""",0µs,10,"""Autumn""","""High""",495850.0,"""Trung cấp""",4.0,"""Mua nhiều"""
"""1771000000002""",1,7559596,2024-10-14 12:13:40.113,155,45000.0,10.71444,0.0,"""In-Store""","""cash""",136d 22h 8m 54s 46ms,10,"""Autumn""","""High""",352361.478357,"""Bình dân""",1.222222,"""Mua ít"""
"""1308000000003""",2,6885875,2024-10-14 16:20:13.623,544,75000.0,11.225257,0.0,"""In-Store""","""card""",257d 20h 51m 1s 654ms,10,"""Autumn""","""High""",384639.619569,"""Bình dân""",1.571429,"""Mua vừa"""


In [39]:
transaction_df = transaction_df.sort("created_date")  # mặc định tăng dần theo thời gian
transaction_df.head()

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level,avg_transaction_amount_per_purchase,segment_name,avg_cat_l1_per_purchase,segment_name_right
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str,f64,str,f64,str
"""2006000000006""",1,4689434,2024-01-01 06:44:59.037,627,35200.0,10.46883,0.451713,"""In-Store""","""cash""",365d 8h 31m 6s 243ms,1,"""Winter""","""High""",1.1825e6,"""Trung cấp""",1.378641,"""Mua ít"""
"""0020010000440""",1,5279260,2024-01-01 06:48:28.537,547,465000.0,13.049795,0.0,"""In-Store""","""cash""",0µs,1,"""Winter""","""High""",465000.0,"""Trung cấp""",1.0,"""Mua ít"""
"""2485000000004""",1,4190229,2024-01-01 06:49:32.443,483,469000.0,13.05836,0.0,"""In-Store""","""cash""",364d 8h 53m 19s 84ms,1,"""Winter""","""High""",511970.68006,"""Trung cấp""",1.162791,"""Mua ít"""
"""2482000000004""",1,6530105,2024-01-01 06:51:11.120,348,525000.0,13.171155,0.0,"""In-Store""","""cash""",360d 7h 56m 39s 227ms,1,"""Winter""","""High""",339501.965496,"""Bình dân""",1.631579,"""Mua vừa"""
"""6767000000003""",1,6393411,2024-01-01 06:52:48.570,560,265000.0,12.487489,0.070175,"""In-Store""","""cash""",301d 12h 1m 4s 490ms,1,"""Winter""","""High""",386749.068627,"""Bình dân""",1.705882,"""Mua vừa"""


#### Train data

In [40]:
from datetime import datetime

begin_hist = datetime(2024, 1, 1)
end_hist = datetime(2024, 11, 30)   # 30/09/2024, không phải 31/09
begin_recent = datetime(2024, 11, 1)
end_recent = datetime(2024, 11, 30)

feature_label_lf = build_feature_label(
    transactions_lf=transaction_df.lazy(),   # convert DataFrame -> LazyFrame
    items_lf=item_df.lazy(),
    users_lf=user_df.lazy(),
    begin_hist=begin_hist,
    end_hist=end_hist,
    begin_recent=begin_recent,
    end_recent=end_recent,
)

feature_label_df = feature_label_lf.collect()
feature_label_df.head()


customer_id,item_id,brand_counts,age_counts,category_counts,segment_counts,target_user_group_counts,time_since_last_purchase_in_B_category,Y
i32,str,u32,u32,u32,u32,u32,i64,i8
6407110,"""6697000000002""",5,5,5,0,5,186,0
4942290,"""3787000000014""",1,3,1,3,9,208,0
6439082,"""7175000000002""",6,25,6,0,19,10,1
4800647,"""5858000000003""",1,0,1,0,2,54,0
6046244,"""1371000000001""",7,10,7,0,7,240,0


In [41]:
feature_label_df['Y'].value_counts()

Y,count
i8,u32
0,20039733
1,2659290


Xử lý negative samples bằng kỹ thuật negative downsampling

In [42]:
df = feature_label_df

In [43]:
import polars as pl

# Tách positive và negative
df_pos = df.filter(pl.col("Y") == 1) 
df_neg = df.filter(pl.col("Y") == 0) 
pos_count = df_pos.height 
neg_target = pos_count * 2 
# tỉ lệ 1:2 
print("Positive =", pos_count) 
print("Negative target =", neg_target) 
# Lấy ngẫu nhiên neg_target mẫu từ negatives 
df_neg_sampled = df_neg.sample(n=neg_target, shuffle=True) 
# Gộp lại 
train_data = pl.concat([df_pos, df_neg_sampled]).sample(fraction=1.0, shuffle=True) 
print(train_data.shape) 
train_data.head()

Positive = 2659290
Negative target = 5318580
(7977870, 9)


customer_id,item_id,brand_counts,age_counts,category_counts,segment_counts,target_user_group_counts,time_since_last_purchase_in_B_category,Y
i32,str,u32,u32,u32,u32,u32,i64,i8
1923522,"""5444000000011""",3,56,5,0,81,196,0
5582484,"""6357000000002""",7,12,1,0,17,12,1
5726182,"""4396000000002""",6,15,6,23,14,20,1
7869009,"""2263000000021""",3,30,3,0,17,3,1
7007188,"""0135000000001""",1,1,1,4,4,130,0


In [44]:
train_data['Y'].value_counts()

Y,count
i8,u32
0,5318580
1,2659290


In [45]:
split_and_save_parquet(train_data, 1, "./", "train")

Đã lưu file: ./sale_pers.train_data_0.parquet


#### Evaluate data

In [46]:
from datetime import datetime

begin_hist = datetime(2024, 1, 1)
end_hist = datetime(2024, 12, 31)
begin_recent = datetime(2024, 12, 1)
end_recent = datetime(2024, 12, 31)

feature_label_lf = build_feature_label(
    transactions_lf=transaction_df.lazy(),   # convert DataFrame -> LazyFrame
    items_lf=item_df.lazy(),
    users_lf=user_df.lazy(),
    begin_hist=begin_hist,
    end_hist=end_hist,
    begin_recent=begin_recent,
    end_recent=end_recent,
)

feature_label_df = feature_label_lf.collect()
feature_label_df.head()


customer_id,item_id,brand_counts,age_counts,category_counts,segment_counts,target_user_group_counts,time_since_last_purchase_in_B_category,Y
i32,str,u32,u32,u32,u32,u32,i64,i8
6711143,"""2278000000028""",4,2,2,8,15,126,0
628682,"""6501000000002""",7,15,6,21,19,250,0
6067581,"""1396000000020""",1,4,1,12,9,276,0
3647946,"""6792000000004""",2,41,2,69,24,179,0
7166679,"""5420000000002""",99,112,99,138,68,135,0


In [47]:
feature_label_df['Y'].value_counts()

Y,count
i8,u32
1,2609159
0,22031908


In [48]:
df = feature_label_df

In [49]:
import polars as pl

# Tách positive và negative
df_pos = df.filter(pl.col("Y") == 1)
df_neg = df.filter(pl.col("Y") == 0)

print("Eval POS (Y=1)  =", df_pos.height)
print("Eval NEG (Y=0)  =", df_neg.height)

# Số negative tối đa mỗi user trong eval
MAX_NEG_PER_USER = 100  # bạn có thể chỉnh: 50, 100, 200,...

# 1) Shuffle toàn bộ negative trước cho random
df_neg_shuffled = df_neg.sample(
    fraction=1.0,
    shuffle=True,
    seed=42,
)

# 2) Với mỗi customer_id, lấy tối đa MAX_NEG_PER_USER negative
df_neg_sampled = (
    df_neg_shuffled
    .group_by("customer_id")
    .head(MAX_NEG_PER_USER)
)

print("Eval NEG sampled (per user, ≤ MAX_NEG_PER_USER) =", df_neg_sampled.height)

# 3) Gộp lại: TẤT CẢ positive + negative đã sample per user
eval_data = pl.concat([df_pos, df_neg_sampled])

# 4) Shuffle toàn bộ eval_data lần nữa cho ngẫu nhiên
eval_data = eval_data.sample(
    fraction=1.0,
    shuffle=True,
    seed=42,
)

print("Eval_data shape:", eval_data.shape)
eval_data.head()


Eval POS (Y=1)  = 2609159
Eval NEG (Y=0)  = 22031908
Eval NEG sampled (per user, ≤ MAX_NEG_PER_USER) = 21750285
Eval_data shape: (24359444, 9)


customer_id,item_id,brand_counts,age_counts,category_counts,segment_counts,target_user_group_counts,time_since_last_purchase_in_B_category,Y
i32,str,u32,u32,u32,u32,u32,i64,i8
3511493,"""0954000000044""",2,2,1,2,2,200,0
1841305,"""4895000000001""",3,36,3,14,25,2,1
5012844,"""5951000000002""",3,5,1,2,11,33,0
5248629,"""0020020000185""",1,0,1,0,2,267,0
7225919,"""4013000000041""",10,4,1,0,26,126,0


In [50]:
split_and_save_parquet(eval_data, 1, "./", "eva")

Đã lưu file: ./sale_pers.eva_data_0.parquet
